In [ ]:
'''
python version 3.10.12
'''

In [ ]:
'''
Make sure to confirm the full path to the requirements.txt file. 
'''
! pip install -r requirements.txt
'''
Please restart this file after executing this cell !!!
'''

In [ ]:
import argparse
import os
import torch
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.nn.functional as F
from dataset import ASVspoof2019
from evaluate_tDCF_asvspoof19 import compute_eer_and_tdcf
from tqdm import tqdm
import eval_metrics as em
import numpy as np

In [3]:
import time
def test_model(feat_model_path, loss_model_path, part, add_loss, device,eval_list,featrues_path,save_path):
    dirname = os.path.dirname
    
    if "checkpoint" in dirname(feat_model_path):
        dir_path = dirname(dirname(feat_model_path))
    else:
        dir_path = dirname(feat_model_path)
    model = torch.load(feat_model_path, map_location="cuda")
    model = model.to(device)
    loss_model = torch.load(loss_model_path) if add_loss != "softmax" else None
   
    test_set = ASVspoof2019("LA", featrues_path,
                            eval_list, part,
                            "LFCC", feat_len=750, padding="repeat")
    testDataLoader = DataLoader(test_set, batch_size=24, shuffle=False, num_workers=0,
                                collate_fn=test_set.collate_fn)
    model.eval()

    with open(save_path, 'w') as cm_score_file:
        use_time=0
        for i, (lfcc,variants, audio_fn, flags,srcs, labels) in enumerate(tqdm(testDataLoader)):
            start_time=time.time()
            lfcc = lfcc.unsqueeze(1).float().to(device)
            # tags = tags.to(device)
            
            labels = labels.to(device)

            feats, lfcc_outputs = model(lfcc)

            score = F.softmax(lfcc_outputs)[:, 0]

            if add_loss == "ocsoftmax":
                loss_model = loss_model.to(device)
             
                ang_isoloss, score = loss_model(feats, labels)
            elif add_loss == "amsoftmax":
                outputs, moutputs = loss_model(feats, labels)
                score = F.softmax(outputs, dim=1)[:, 0]
            end_time=time.time()
            use_time+=end_time-start_time
            for j in range(labels.size(0)):
                cm_score_file.write(
                    '%s %s %s %s %s %s\n' % (variants[j],audio_fn[j], flags[j],srcs[j],
                                          "spoof" if labels[j].data.cpu().numpy() else "bonafide",
                                          score[j].item()))
        print(f'use time:{use_time}s')
   

def test( add_loss, device,model_path,loss_model_path,eval_list,featrues_path,save_path):
    
    
    # print(device)
    test_model(model_path, loss_model_path, "eval", add_loss, device,eval_list,featrues_path,save_path)

def test_individual_attacks(cm_score_file):
    asv_score_file = os.path.join('/data/neil/DS_10283_3336',
                                  'LA/ASVspoof2019_LA_asv_scores/ASVspoof2019.LA.asv.eval.gi.trl.scores.txt')

    # Fix tandem detection cost function (t-DCF) parameters
    Pspoof = 0.05
    cost_model = {
        'Pspoof': Pspoof,  # Prior probability of a spoofing attack
        'Ptar': (1 - Pspoof) * 0.99,  # Prior probability of target speaker
        'Pnon': (1 - Pspoof) * 0.01,  # Prior probability of nontarget speaker
        'Cmiss_asv': 1,  # Cost of ASV system falsely rejecting target speaker
        'Cfa_asv': 10,  # Cost of ASV system falsely accepting nontarget speaker
        'Cmiss_cm': 1,  # Cost of CM system falsely rejecting target speaker
        'Cfa_cm': 10,  # Cost of CM system falsely accepting spoof
    }

    # Load organizers' ASV scores
    asv_data = np.genfromtxt(asv_score_file, dtype=str)
    asv_sources = asv_data[:, 0]
    asv_keys = asv_data[:, 1]
    asv_scores = asv_data[:, 2].astype(np.float)

    # Load CM scores
    cm_data = np.genfromtxt(cm_score_file, dtype=str)
    cm_utt_id = cm_data[:, 0]
    cm_sources = cm_data[:, 1]
    cm_keys = cm_data[:, 2]
    cm_scores = cm_data[:, 3].astype(np.float)

    other_cm_scores = -cm_scores

    eer_cm_lst, min_tDCF_lst = [], []
    for attack_idx in range(7,20):
        # Extract target, nontarget, and spoof scores from the ASV scores
        tar_asv = asv_scores[asv_keys == 'target']
        non_asv = asv_scores[asv_keys == 'nontarget']
        spoof_asv = asv_scores[asv_sources == 'A%02d' % attack_idx]

        # Extract bona fide (real human) and spoof scores from the CM scores
        bona_cm = cm_scores[cm_keys == 'bonafide']
        spoof_cm = cm_scores[cm_sources == 'A%02d' % attack_idx]

        # EERs of the standalone systems and fix ASV operating point to EER threshold
        eer_asv, asv_threshold = em.compute_eer(tar_asv, non_asv)
        eer_cm = em.compute_eer(bona_cm, spoof_cm)[0]

        other_eer_cm = em.compute_eer(other_cm_scores[cm_keys == 'bonafide'], other_cm_scores[cm_sources == 'A%02d' % attack_idx])[0]

        [Pfa_asv, Pmiss_asv, Pmiss_spoof_asv] = em.obtain_asv_error_rates(tar_asv, non_asv, spoof_asv, asv_threshold)

        if eer_cm < other_eer_cm:
            # Compute t-DCF
            tDCF_curve, CM_thresholds = em.compute_tDCF(bona_cm, spoof_cm, Pfa_asv, Pmiss_asv, Pmiss_spoof_asv, cost_model,
                                                        True)
            # Minimum t-DCF
            min_tDCF_index = np.argmin(tDCF_curve)
            min_tDCF = tDCF_curve[min_tDCF_index]

        else:
            tDCF_curve, CM_thresholds = em.compute_tDCF(other_cm_scores[cm_keys == 'bonafide'],
                                                        other_cm_scores[cm_sources == 'A%02d' % attack_idx],
                                                        Pfa_asv, Pmiss_asv, Pmiss_spoof_asv, cost_model, True)
            # Minimum t-DCF
            min_tDCF_index = np.argmin(tDCF_curve)
            min_tDCF = tDCF_curve[min_tDCF_index]
        eer_cm_lst.append(min(eer_cm, other_eer_cm))
        min_tDCF_lst.append(min_tDCF)

    return eer_cm_lst, min_tDCF_lst

In [ ]:

if __name__ == "__main__":
   
    model_path="change to the path to OC-Softmax model" # download here[https://huggingface.co/VoiceWukong/VoiceWukong/resolve/main/OC-Softmax.pt?download=true]
    loss_model_path='./anti-spoofing_loss_model.pt'
    zh_eval_list_path='change to the path to zh_eval_list.txt'
    '''
    |- ZH_Features
          |- LFCC_Alldataset
          |- LFCC_Alldataset32K
          |- ....
    '''
    zh_features_path='change to the path to the ZH_Features/'
    zh_save_path='change to the path to save zh_eval_score.txt'
    en_eval_list_path='change to the path to eval_list.txt'
    en_features_path='change to the path to the Features/'
    en_save_path='change to the path to save en_eval_score.txt'
    
    loss='ocsoftmax'
    os.environ["CUDA_VISIBLE_DEVICES"] = "1"
    # torch.cuda.set_device(1)
   
    
       
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"device: {device}")
    test(loss, 
         device,
         model_path,
         loss_model_path,
         eval_list=zh_eval_list_path,
         featrues_path=zh_features_path,
         save_path=zh_save_path)
    test(loss, 
         device,
         model_path,
         loss_model_path,
         eval_list=en_eval_list_path,
         featrues_path=en_features_path,
         save_path=en_save_path)